In [ ]:
# =============================================================================
# jobs/30_build_hazard_rasters.py
# Flood hazard for the exposure grid: paints the Environment Agency flood polygons onto 10 m rasters, one per 50 km tile, and
# summarises them as the share of each 100 m cell in each band (plan: docs/EXPOSURE_AND_RISK_GRID_PLAN.md).
#
# SOURCES (governed volumes, present-day folders only; folders with Climate_Change in the name are skipped)
#   rofrs         Risk of Flooding from Rivers and Sea, the four-band file per tile       bands Very Low, Low, Medium, High
#   rofsw         Risk of Flooding from Surface Water, the banded extent file per tile    bands Very Low, Low, Medium, High
#                 NOT part of source = all. It is the largest source (about 85 GB with its depth files) and surface water is not what
#                 warning areas are issued for. Run it on its own, later, for the context under the FGS surface water risk polygons.
#   flood_zones   Flood Map for Planning, Flood Zones 2 and 3 (one national file)         bands FZ2, FZ3 (FZ1 is everywhere else)
#
# COASTAL, FLUVIAL AND SURFACE ARE DIFFERENT THINGS, and are never merged here:
#   rofrs is rivers AND sea together; the data does not split it. Read it with the warning area's type (fluvial, coastal, tidal).
#   rofsw is surface water, a separate hazard. It is not what flood warning areas are issued for.
#   flood_zones carries a flood_source, so it is also written split: flood_zones_sea (coastal), flood_zones_river (fluvial),
#   flood_zones_river_and_sea (mapped together, cannot be split) and flood_zones (all of them).
#
# OUTPUT  config.HAZARD_RASTER_DIR/<source>/<tile>.npz   uint8 raster, 0 no band, 1 to 4 the band rank (worst band wins at a pixel)
#         config.TBL_HAZARD_CELLS    source, tile, cell_key, band, share   sparse: only cells with some band, shares add to at most 1
# Job 33 samples the rasters at address points; the cell table feeds the 100 m grid and areas with no addresses.
#
# RESUME  A tile is finished when its .npz exists, and the table rows are written first. With resume = yes a re-run skips finished
# tiles; with resume = no a source starts again from nothing.
#
# WHAT TO CHECK (printed at the end): tiles done and failed; flooded area in km2 by band against the England area (130,000 km2);
# High should be a few per cent or less of England for either source, and Very Low the largest band.
#
# SPEED  Tiles are independent, so workers > 1 runs that many at once on a thread pool. On a single node with 16 cores and 64 GB
# (Standard_D16ds_v5, no GPU, no machine learning runtime) try 8. Each worker holds a few hundred MB, more for the biggest tiles.
# On serverless leave it at 1 or 2.
#
# Widgets: source, tiles, resume, workers, res_m, rofrs_dir, rofsw_dir, flood_zones_file, write_table.
# =============================================================================

import os
import sys

_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

# shapely and rasterio are not on every cluster. If either has to be installed the notebook restarts Python: run all again once.
try:
    import rasterio  # noqa: F401
    import shapely  # noqa: F401
except ImportError:
    get_ipython().run_line_magic("pip", "install -q shapely rasterio")
    dbutils.library.restartPython()

import importlib
import shutil
import tempfile
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import shapely

import config
import importlib as _il

_il.reload(config)   # a Python session that ran an earlier version of config.py would otherwise miss tables added since
import utils.hazard_raster as _hz
importlib.reload(_hz)
from utils.grid100 import band_rank
from utils.hazard_raster import RasterAccumulator, cell_shares, is_present_day, save_raster, tile_bounds, tile_code
from utils.helpers import get_spark

spark = get_spark()

D = "/Volumes/prd_dash_bronze/defra_data_services_platform_unrestricted"
dbutils.widgets.dropdown("source", "rofrs", ["rofrs", "flood_zones", "all", "rofsw"], "Source (all = rivers and sea plus Flood Zones; surface water only when chosen)")
dbutils.widgets.text("tiles", "", "Only these tiles, for a trial (comma separated, e.g. SE50,SE55). Blank is all")
dbutils.widgets.dropdown("resume", "yes", ["yes", "no"], "Skip tiles already done")
dbutils.widgets.text("workers", "1", "Tiles to run at once (8 on a 16 core node)")
dbutils.widgets.text("res_m", str(config.HAZARD_RES_M), "Raster resolution in metres (must divide 100)")
dbutils.widgets.text("rofrs_dir", f"{D}/rof_rivers_sea/format_GEOPARQUET_rof_rivers_sea/LATEST_rof_rivers_sea/", "Rivers and sea folder")
dbutils.widgets.text("rofsw_dir", f"{D}/rof_surface_water/format_GEOPARQUET_rof_surface_water/LATEST_rof_surface_water/", "Surface water folder")
dbutils.widgets.text("flood_zones_file", f"{D}/ea_flood_map_for_planning_flood_zones/format_GEOPARQUET_ea_flood_map_for_planning_flood_zones/LATEST_ea_flood_map_for_planning_flood_zones/", "Flood Zones folder")
dbutils.widgets.dropdown("write_table", "yes", ["yes", "no"], "Write the cell table")

SOURCE = dbutils.widgets.get("source")
ONLY = [t.strip().upper() for t in dbutils.widgets.get("tiles").split(",") if t.strip()]
RESUME = dbutils.widgets.get("resume") == "yes"
WORKERS = max(1, int(dbutils.widgets.get("workers") or 1))
RES = int(dbutils.widgets.get("res_m") or config.HAZARD_RES_M)
WRITE_TABLE = dbutils.widgets.get("write_table") == "yes"
if 100 % RES:
    raise ValueError("res_m must divide 100")

NAFRA_LABELS = {1: "Very Low", 2: "Low", 3: "Medium", 4: "High"}
SOURCES = {
    "rofrs": {"dir": dbutils.widgets.get("rofrs_dir"), "file_start": "RoFRS_4band", "band_col": "Risk_band", "labels": NAFRA_LABELS,
              "rank": lambda v: band_rank(v)},
    "rofsw": {"dir": dbutils.widgets.get("rofsw_dir"), "file_start": "RoFSW.parquet", "band_col": "Risk_band", "labels": NAFRA_LABELS,
              "rank": lambda v: band_rank(v)},
    "flood_zones": {"dir": dbutils.widgets.get("flood_zones_file"), "band_col": "flood_zone", "source_col": "flood_source", "labels": {1: "FZ2", 2: "FZ3"},
                    "rank": lambda v: {"FZ2": 1, "FZ3": 2}.get(str(v).strip().upper(), 0)},
}
os.makedirs(config.HAZARD_RASTER_DIR, exist_ok=True)
print(f"Source: {SOURCE}. Workers: {WORKERS}. Resolution {RES} m. Resume: {RESUME}. Trial tiles: {ONLY or 'none (all)'}")


In [ ]:
# =============================================================================
# 1. THE TILES: present-day folders only, and the one file in each that holds the bands
# =============================================================================

def local(path):
    return path.replace("dbfs:", "")


def list_tiles(source):
    """{tile: parquet path} for the present-day tile folders of a tiled source."""
    cfg = SOURCES[source]
    out = {}
    for d in dbutils.fs.ls(cfg["dir"]):
        if not d.name.endswith("/") or not is_present_day(d.name):
            continue
        code = tile_code(d.name)
        if code is None:
            continue
        files = [f.name for f in dbutils.fs.ls(d.path) if f.name.endswith(".parquet")]
        # the band file: the name starts as expected and is not a depth threshold
        match = [f for f in files if f.startswith(cfg["file_start"]) and "depth" not in f]
        if len(match) != 1:
            print(f"   {d.name}: expected one band file, found {match or files}. Skipped.")
            continue
        out[code] = local(d.path) + match[0]
    return dict(sorted(out.items()))


tiles_by_source = {}
for s in (["rofrs"] if SOURCE == "all" else [SOURCE]):
    if s != "flood_zones":
        tiles_by_source[s] = list_tiles(s)
        print(f"{s}: {len(tiles_by_source[s])} present-day tiles")
if SOURCE in ("flood_zones", "all"):
    # the Flood Zones file is national: use the rivers and sea tile list as the England tile list
    tiles_by_source["flood_zones"] = {t: SOURCES["flood_zones"]["dir"] for t in (tiles_by_source.get("rofrs") or list_tiles("rofrs"))}
    print(f"flood_zones: one national file, read once per tile ({len(tiles_by_source['flood_zones'])} tiles)")
    fs = spark.read.parquet(SOURCES["flood_zones"]["dir"]).groupBy("flood_zone", "flood_source").count().toPandas()
    print("Flood Zone polygons by zone and mapping source (this decides the coastal and fluvial split):")
    display(fs.astype(str))
if ONLY:
    tiles_by_source = {s: {t: p for t, p in d.items() if t in ONLY} for s, d in tiles_by_source.items()}
    print("Trial tiles:", {s: list(d) for s, d in tiles_by_source.items()})


In [ ]:
# =============================================================================
# 2. PAINT EACH TILE, SUMMARISE IT, AND WRITE IT
# Polygons are read in batches, so a tile bigger than memory still works. The table rows are written before the .npz, so a tile
# with an .npz is complete.
# =============================================================================

def raster_path(source, tile):
    return f"{config.HAZARD_RASTER_DIR}/{source}/{tile}.npz"


def paint_tiled_file(path, cfg, bounds):
    """RoFRS and RoFSW: a parquet file of band polygons for one tile, read in row batches."""
    acc = RasterAccumulator(bounds, RES, ranks=tuple(cfg["labels"]))
    pf = pq.ParquetFile(path)
    for batch in pf.iter_batches(batch_size=20000, columns=[cfg["band_col"], "geometry"]):
        d = batch.to_pandas()
        acc.add(d["geometry"].to_numpy(), [cfg["rank"](v) for v in d[cfg["band_col"]]])
    return acc


def fz_group(flood_source):
    """The mapping source of a Flood Zone polygon: sea (coastal), river (fluvial), river_and_sea (mapped together) or other."""
    t = str(flood_source).lower()
    sea = any(w in t for w in ("sea", "coast", "tidal"))
    river = any(w in t for w in ("river", "fluvial"))
    return "river_and_sea" if sea and river else "sea" if sea else "river" if river else "other"


FZ_OUTPUTS = ("flood_zones", "flood_zones_sea", "flood_zones_river", "flood_zones_river_and_sea", "flood_zones_other")


def paint_flood_zones(cfg, bounds):
    """Flood Zones: one national file; only polygons whose bounding box meets the tile are read. Returns {output name: accumulator}:
    all polygons together, and one raster for each mapping source, so coastal and fluvial can be told apart."""
    xmin, ymin, xmax, ymax = bounds
    df = (spark.read.parquet(cfg["dir"])
          .filter(f"bbox.xmax >= {xmin} AND bbox.xmin <= {xmax} AND bbox.ymax >= {ymin} AND bbox.ymin <= {ymax}")
          .select(cfg["band_col"], cfg["source_col"], "geometry"))
    pdf = df.toPandas()
    ranks = np.array([cfg["rank"](v) for v in pdf[cfg["band_col"]]])
    groups = np.array([fz_group(v) for v in pdf[cfg["source_col"]]])
    out = {}
    for name in FZ_OUTPUTS:
        sel = np.ones(len(pdf), dtype=bool) if name == "flood_zones" else groups == name.replace("flood_zones_", "")
        # every output gets a raster, empty if it has no polygons here, so a finished tile is recognised on a re-run
        acc = RasterAccumulator(bounds, RES, ranks=tuple(cfg["labels"]))
        if sel.any():
            acc.add(pdf["geometry"].to_numpy()[sel], ranks[sel])
        out[name] = acc
    return out


def clear_source_rows(source):
    if not WRITE_TABLE:
        return
    try:
        spark.sql(f"DELETE FROM {config.TBL_HAZARD_CELLS} WHERE source = '{source}'")
    except Exception as e:
        if "TABLE_OR_VIEW_NOT_FOUND" not in str(e) and "cannot be found" not in str(e).lower():
            raise


def write_cells(source, tile, shares):
    # a tile with no flooded cells (open sea, say) has nothing to write; Spark cannot infer types from an empty frame
    if not WRITE_TABLE or shares.empty:
        return
    out = shares.assign(source=source, tile=tile)[["source", "tile", "cell_key", "band", "share"]]
    out["share"] = out["share"].astype("float32")
    sdf = spark.createDataFrame(out)
    (sdf.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(config.TBL_HAZARD_CELLS))


def finish_tile(source, tile, acc, cfg, bounds, t0, i, n):
    """Summarise one painted tile, write its rows, then its raster (the raster marks the tile as done)."""
    arr = acc.finalize()
    shares = cell_shares(arr, bounds, RES, cfg["labels"])
    out_path = raster_path(source, tile)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    if (RESUME or ONLY) and WRITE_TABLE:
        try:
            spark.sql(f"DELETE FROM {config.TBL_HAZARD_CELLS} WHERE source = '{source}' AND tile = '{tile}'")
        except Exception:
            pass
    write_cells(source, tile, shares)
    # a volume takes sequential writes only, and an .npz is written with seeks, so build it locally and copy it across
    with tempfile.TemporaryDirectory() as tmp:
        tmp_file = os.path.join(tmp, f"{source}_{tile}.npz")
        save_raster(tmp_file, arr, bounds, RES)
        shutil.copyfile(tmp_file, out_path)
    km2 = {cfg["labels"][r]: round(float((arr == r).sum()) * RES * RES / 1e6, 1) for r in cfg["labels"]}
    log.append({"source": source, "tile": tile, "status": "done", "polygons": acc.n_polygons, "cells": len(shares),
                "seconds": round(time.time() - t0, 1), **{f"km2_{k}": v for k, v in km2.items()}})
    print(f"[{i}/{n}] {source} {tile}: {acc.n_polygons:,} polygons, {len(shares):,} cell rows, {time.time() - t0:.0f}s")


log = []
log_lock = threading.Lock()


def process(source, tile, path, i, n):
    cfg = SOURCES[source]
    outputs = FZ_OUTPUTS if source == "flood_zones" else (source,)
    if RESUME and all(os.path.exists(raster_path(o, tile)) for o in outputs):
        log.append({"source": source, "tile": tile, "status": "skipped (done)"})
        return
    t0 = time.time()
    try:
        bounds = tile_bounds(tile)
        if source == "flood_zones":
            for name, acc in paint_flood_zones(cfg, bounds).items():
                finish_tile(name, tile, acc, cfg, bounds, t0, i, n)
        else:
            finish_tile(source, tile, paint_tiled_file(path, cfg, bounds), cfg, bounds, t0, i, n)
    except Exception as e:
        log.append({"source": source, "tile": tile, "status": "FAILED: " + str(e)[:150]})
        print(f"[{i}/{n}] {source} {tile}: FAILED {str(e)[:150]}")


for source, tiles in tiles_by_source.items():
    outputs = FZ_OUTPUTS if source == "flood_zones" else (source,)
    if not RESUME and not ONLY:
        for o in outputs:
            clear_source_rows(o)
    work = [(source, tile, path, i, len(tiles)) for i, (tile, path) in enumerate(tiles.items(), 1)]
    if WORKERS == 1:
        for w in work:
            process(*w)
    else:
        with ThreadPoolExecutor(max_workers=WORKERS) as pool:
            for f in as_completed([pool.submit(process, *w) for w in work]):
                f.result()


In [ ]:
# =============================================================================
# 3. CHECKS
# =============================================================================

log_df = pd.DataFrame(log)
print("Tiles by status:")
display(log_df.groupby(["source", "status"]).size().rename("tiles").reset_index().astype(str))
failed = log_df[log_df["status"].str.startswith("FAILED")] if len(log_df) else log_df
if len(failed):
    print("Failed tiles (re-run with resume = yes to retry only these):")
    display(failed.astype(str))

done = log_df[log_df["status"] == "done"] if len(log_df) else log_df
if len(done):
    km2_cols = [c for c in done.columns if c.startswith("km2_")]
    summary = done.groupby("source")[km2_cols].sum().round(0)
    print("\nFlooded area this run in km2, by band and by hazard (rivers and sea, surface water and the Flood Zone splits are separate rows) (England is about 130,000 km2; High should be small, Very Low the largest):")
    display(summary.reset_index().astype(str))
    print("\nSlowest tiles:")
    display(done.sort_values("seconds", ascending=False).head(5)[["source", "tile", "polygons", "cells", "seconds"]].astype(str))

if WRITE_TABLE and len(done):
    spark.sql(f"""COMMENT ON TABLE {config.TBL_HAZARD_CELLS} IS 'Share of each 100 m British National Grid cell in each flood band, from the Environment Agency Risk of Flooding from Rivers and Sea (rofrs), Surface Water (rofsw) and Flood Zones 2 and 3 (flood_zones), present day, painted at 10 m. Sparse: only cells with some band. Job 30.'""")
    total = spark.table(config.TBL_HAZARD_CELLS).groupBy("source").count().toPandas()
    print("\nRows now in the cell table:")
    display(total.astype(str))


In [ ]:
# Exit is in its own cell: in an interactive run Databricks replaces a cell's output with the exit message,
# so the checks above would be hidden if this line shared their cell.
dbutils.notebook.exit("success" if not len(failed) else "finished with failed tiles")
